# E5 — Maximum stable coupling stiffness

`e5_kmax.py` appends every K step to `kmax.csv`; K_max of a ladder (method, delay, grip) is its last
stable K before the first unstable one. Expected: flat for the model-based methods (set by the device
and the 1 kHz loop), falling with T for M1. M3–M5 should give nearly the same K_max.

In [ ]:
KMAX_CSV = "~/data/franka/rim-delays/robot/e5/kmax.csv"

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
sys.path[:0] = [str(HERE), str(HERE.parent)]
import matplotlib.pyplot as plt  # noqa: I001
import pandas as pd

import plots as P

%matplotlib inline
%load_ext autoreload
%autoreload 2  # pick up edits to exp_metrics / plots / common without a kernel restart
steps = pd.read_csv(Path(KMAX_CSV).expanduser())
steps["T"] = steps.rtt.map(P.delay_key)
steps

In [ ]:
def k_max(g):
    g = g.sort_values("K")
    unstable = g[~g.stable.astype(bool)]
    ok = g[g.K < unstable.K.min()] if len(unstable) else g
    return ok.K.max() if len(ok) else float("nan")


kmax = steps.groupby(["method", "T", "grip"]).apply(k_max).rename("K_max").reset_index()
kmax

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
xs = P.delay_axis(ax, kmax["T"])
for m in P.methods_in(kmax):
    for grip, ls in (("held", "-"), ("released", "--")):
        d = kmax[(kmax.method == m) & (kmax.grip == grip)].set_index("T").reindex([k for k in xs])
        ax.plot([xs[k] for k in d.index], d.K_max, "o" + ls, color=P.color(m), label=f"{P.LABELS.get(m, m)}, {grip}")
ax.set_ylabel("K_max [N/m]")
ax.legend(fontsize=8)
fig.tight_layout()